# Fase 1 — Análisis Exploratorio de Datos (EDA)
## Detección de Fraude con Tarjetas de Crédito

**Proyecto MLOps — Semana 14 (Software Inteligente, UNMSM)**
**Fase 1 (Data) · Responsable: Integrante 1**

Este notebook contiene **únicamente el trabajo técnico de exploración de la Fase 1**: carga del dataset, verificación de calidad (nulos y duplicados), distribución de clases, análisis de `Amount` por clase y matriz de correlación. Genera y guarda los gráficos en `/reports`.

> La **interpretación de los resultados, los hallazgos y las decisiones de datos** NO van en este notebook: se documentan aparte en **[`reports/hallazgos_fase1.md`](../reports/hallazgos_fase1.md)**, que se entrega junto con este notebook a quien continúe con la Fase 2 (Modeling).

## 0. Configuración e importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Estilo de gráficos
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

# Carpeta para guardar los gráficos que irán al informe (/reports)
REPORTS_DIR = 'reports'
os.makedirs(REPORTS_DIR, exist_ok=True)

print('Librerías cargadas correctamente.')

## 1. Carga del dataset

La celda de abajo busca `creditcard.csv` automáticamente.

- **En local:** lo busca en `/data`.
- **En Colab (incl. Colab remoto desde VSCode):** monta tu **Google Drive** y lee el archivo desde ahí. Para que funcione, primero **sube `creditcard.csv` a tu Google Drive** (a la raíz "Mi unidad"). Al correr la celda te pedirá autorizar el montaje del Drive.

> El botón de subida manual (`files.upload()`) **no funciona** cuando Colab se usa de forma remota desde VSCode; por eso se usa Google Drive.

In [ ]:
# Detecta si estamos en Colab
try:
    import google.colab  # noqa
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# Rutas candidatas donde podría estar el CSV (local, notebook en la raíz del repo)
rutas = ['creditcard.csv', 'data/creditcard.csv', '../creditcard.csv']
ruta_csv = next((r for r in rutas if os.path.exists(r)), None)

if ruta_csv is None and EN_COLAB:
    # En Colab (incl. Colab remoto desde VSCode) montamos Google Drive.
    # NOTA: el botón files.upload() NO funciona con Colab remoto en VSCode,
    # por eso usamos Drive. Sube 'creditcard.csv' a tu Drive antes de correr.
    from google.colab import drive
    drive.mount('/content/drive')
    # Busca el CSV en las ubicaciones típicas de tu Drive
    rutas_drive = [
        '/content/drive/MyDrive/octavo/creditcard.csv',
        '/content/drive/MyDrive/creditcard.csv',
        '/content/drive/MyDrive/data/creditcard.csv',
        '/content/drive/MyDrive/tarea-semana14/data/creditcard.csv',
    ]
    ruta_csv = next((r for r in rutas_drive if os.path.exists(r)), None)

assert ruta_csv is not None, (
    'No se encontró creditcard.csv. En Colab: súbelo a tu Google Drive '
    '(raíz "Mi unidad") y vuelve a correr esta celda. En local: colócalo junto al notebook o en /data.'
)
print(f'Cargando desde: {ruta_csv}')
df = pd.read_csv(ruta_csv)
print('Dataset cargado.')

## 2. Descripción básica y calidad de los datos

Tamaño, tipos de datos y verificación (con código) de valores nulos y filas duplicadas.

In [ ]:
print(f'Filas x Columnas: {df.shape[0]:,} x {df.shape[1]}')
print(f'\nColumnas: {list(df.columns)}')
df.head()

In [ ]:
# Tipos de datos y memoria
df.info()

In [ ]:
# ---- Valores nulos (demostrado con código) ----
nulos_totales = df.isnull().sum().sum()
print(f'Total de valores nulos en todo el dataset: {nulos_totales}')
if nulos_totales == 0:
    print('=> No hay valores faltantes. El dataset está completo.')
else:
    print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# ---- Filas duplicadas (demostrado con código) ----
n_dup = df.duplicated().sum()
print(f'Filas duplicadas exactas: {n_dup:,}')
print(f'Porcentaje del total: {n_dup / len(df) * 100:.3f}%')

In [ ]:
# El análisis se realiza sobre el dataset sin duplicados exactos
filas_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Filas antes: {filas_antes:,}  ->  Filas después: {len(df):,}')
print(f'Duplicados eliminados: {filas_antes - len(df):,}')

## 3. Distribución de la clase (`Class`)

Cantidad y porcentaje de transacciones fraudulentas vs. legítimas.

In [ ]:
conteo = df['Class'].value_counts().sort_index()
porcentaje = df['Class'].value_counts(normalize=True).sort_index() * 100

resumen_clase = pd.DataFrame({
    'Clase': ['0 - Legítima', '1 - Fraude'],
    'Cantidad': conteo.values,
    'Porcentaje (%)': porcentaje.round(3).values
})
print(resumen_clase.to_string(index=False))

n_fraude = int(conteo[1]); n_legit = int(conteo[0])
print(f'\n=> 1 fraude por cada ~{n_legit // n_fraude} transacciones legítimas.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Escala normal (se ve lo extremo del desbalance)
sns.countplot(x='Class', data=df, ax=axes[0], palette=['#2e86de', '#e74c3c'])
axes[0].set_title('Distribución de clases (escala normal)')
axes[0].set_xticklabels(['Legítima (0)', 'Fraude (1)'])
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=9)

# Escala logarítmica (permite ver la barra de fraude)
sns.countplot(x='Class', data=df, ax=axes[1], palette=['#2e86de', '#e74c3c'])
axes[1].set_yscale('log')
axes[1].set_title('Distribución de clases (escala logarítmica)')
axes[1].set_xticklabels(['Legítima (0)', 'Fraude (1)'])

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/01_desbalance_clases.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado en /reports/01_desbalance_clases.png')

## 4. Distribución de `Amount` (monto) separada por clase

Los montos de fraude suelen comportarse distinto. Lo verificamos con estadísticas y con un boxplot.

In [ ]:
resumen_amount = df.groupby('Class')['Amount'].describe()[['mean', '50%', 'std', 'max']]
resumen_amount.index = ['Legítima (0)', 'Fraude (1)']
resumen_amount.columns = ['Media', 'Mediana', 'Desv. Est.', 'Máximo']
print('Estadísticas de Amount por clase:')
print(resumen_amount.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Boxplot de Amount por clase (recortamos eje Y para que sea legible)
sns.boxplot(x='Class', y='Amount', data=df, ax=axes[0], palette=['#2e86de', '#e74c3c'])
axes[0].set_ylim(0, 400)
axes[0].set_title('Amount por clase (boxplot, recortado a $400)')
axes[0].set_xticklabels(['Legítima (0)', 'Fraude (1)'])

# Histogramas superpuestos
axes[1].hist(df[df.Class == 0]['Amount'], bins=50, range=(0, 500), alpha=0.6, label='Legítima', color='#2e86de', density=True)
axes[1].hist(df[df.Class == 1]['Amount'], bins=50, range=(0, 500), alpha=0.6, label='Fraude', color='#e74c3c', density=True)
axes[1].set_title('Distribución de Amount (densidad, hasta $500)')
axes[1].set_xlabel('Amount'); axes[1].legend()

plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/02_amount_por_clase.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado en /reports/02_amount_por_clase.png')

## 5. Matriz de correlación con la clase

Identificamos qué variables `V1`–`V28` (y `Amount`, `Time`) se relacionan más con el fraude. Como son muchas variables, mostramos primero la matriz completa y luego el ranking de las más correlacionadas con `Class`.

In [ ]:
corr = df.corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True,
            cbar_kws={'shrink': 0.7}, xticklabels=True, yticklabels=True)
plt.title('Matriz de correlación (todas las variables)')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/03_matriz_correlacion.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado en /reports/03_matriz_correlacion.png')

In [ ]:
# Ranking: variables más correlacionadas con Class (valor absoluto)
corr_class = corr['Class'].drop('Class').sort_values(key=abs, ascending=False)
print('Top 10 variables más correlacionadas con el fraude (Class):')
print(corr_class.head(10).round(3).to_string())

# Gráfico de barras del top
plt.figure(figsize=(9, 5))
top = corr_class.head(10)
colores = ['#e74c3c' if v > 0 else '#2e86de' for v in top.values]
plt.barh(top.index[::-1], top.values[::-1], color=colores[::-1])
plt.axvline(0, color='gray', lw=0.8)
plt.title('Top 10 correlaciones con Class (rojo=positiva, azul=negativa)')
plt.xlabel('Correlación con Class')
plt.tight_layout()
plt.savefig(f'{REPORTS_DIR}/04_top_correlaciones.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado en /reports/04_top_correlaciones.png')